# Hafta 1 — Temeller: Dil, Olasılık ve Bilgi

Tek bir dizeyi tokenlara ayırmadan önce NLP'yi üç disipline dayandırıyoruz: kuramsal dilbilim, olasılık kuramı ve bilgi kuramı. Bu hafta, dilin bir makine için *ne* olduğunu ve olasılıksal modellemenin *neden* doğru çerçeve olduğunu yanıtlıyor.

## Öğrenme Hedefleri

Bu haftanın sonunda şunları yapabileceksiniz:

- Biçimbilim (morfoloji), sözdizimi, anlambilim ve edimbilimi birbirinden ayırt etmek ve her birinin modern NLP sistemlerinde nerede ortaya çıktığını belirlemek.
- Entropi, çapraz entropi, KL ıraksaması ve karşılıklı bilgiyi temel ilkelerden türetmek.
- Gerçek bir derlem üzerinde İngilizcenin deneysel entropisini hesaplamak ve sonucu Shannon (1948) ışığında yorumlamak.
- Kalan 11 hafta için tekrarlanabilir bir Python ortamı kurmak.

## Zorunlu Okumalar

- Shannon, C. E. (1948). *A Mathematical Theory of Communication*.
- Manning, C. D. ve Schütze, H. (1999). *Foundations of Statistical NLP*, Bölüm 1–2.
- Jurafsky, D. ve Martin, J. H. (2024). *Speech and Language Processing*, 3. baskı, Bölüm 1.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import math
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)

## 1. Bir makine için dil nedir?

Dil hiyerarşiktir (sesbirim → biçimbirim → sözcük → öbek → cümle → söylem), üretkendir (sonlu kurallar sonsuz sayıda söz üretir) ve her yerde belirsizlik içerir. Bir makine yalnızca bir bayt akışı görür; bunun üzerindeki her yapı düzeyini açıkça modellememiz gerekir.

Klasik belirsiz cümleyi ele alalım (örnek İngilizcedir, çünkü belirsizlik İngilizceye özgüdür):

> *Time flies like an arrow; fruit flies like a banana.*
> ("Zaman bir ok gibi uçar; meyve sinekleri muzu sever.")

**flies** sözcükbirimi ilk yan cümlede bir fiil, ikincisinde bir isimdir; **like** ilkinde bir edat, ikincisinde bir fiildir. Hiçbir karakter torbası istatistiği bunu çözemez — sözdizimsel ve anlamsal bağlama ihtiyacımız vardır. Dersin geri kalanı, bir anlamda, giderek zenginleşen bağlam pencereleri kurmanın hikâyesidir.

In [ ]:
sentence = "Time flies like an arrow; fruit flies like a banana."
analysis = {
    'flies (1. yan cümle)': ('fiil', 'havada hareket etmek'),
    'flies (2. yan cümle)': ('isim', 'Diptera takımından küçük böcekler (sinekler)'),
    'like (1. yan cümle)':  ('edat', '-gibi, -e benzer biçimde'),
    'like (2. yan cümle)':  ('fiil', 'sevmek, hoşlanmak'),
}
for token, (pos, gloss) in analysis.items():
    print(f"{token:<24} {pos:<6} {gloss}")

## 2. Olasılık: belirsizliğin dili

Dört yapıyı sürekli kullanırız:

**Koşullu olasılık.**  $P(A \mid B) = P(A, B) / P(B)$.

**Zincir kuralı.** $w_1, \ldots, w_T$ token dizisi için:

$$P(w_1, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_{<t}).$$

Bu özdeşlik, derste karşılaşacağımız her dil modelinin omurgasıdır.

**Bayes teoremi.** $P(H \mid D) \propto P(D \mid H) P(H)$.

**En çok olabilirlik (maximum likelihood).** $\prod_i P(x_i \mid \theta)$ ifadesini — eşdeğer olarak log-olabilirlik $\sum_i \log P(x_i \mid \theta)$ ifadesini — en büyükleyen $\theta$ seçilir.

In [ ]:
# Oyuncak bir dağılımla 3 sözcüklük bir dizi üzerinde küçük bir zincir kuralı gösterimi.
# Ortak olasılık = P(w1) * P(w2|w1) * P(w3|w1,w2)
p_w1     = {'the': 0.6, 'a': 0.4}
p_w2_w1  = {'the': {'cat': 0.5, 'dog': 0.5},
            'a':   {'cat': 0.7, 'dog': 0.3}}
p_w3_w12 = {('the','cat'): {'sat': 0.8, 'ran': 0.2},
            ('the','dog'): {'sat': 0.3, 'ran': 0.7},
            ('a','cat'):   {'sat': 0.6, 'ran': 0.4},
            ('a','dog'):   {'sat': 0.2, 'ran': 0.8}}

def joint(w1, w2, w3):
    return p_w1[w1] * p_w2_w1[w1][w2] * p_w3_w12[(w1,w2)][w3]

total = sum(joint(a,b,c) for a in p_w1 for b in p_w2_w1[a] for c in p_w3_w12[(a,b)])
print(f"P(the,cat,sat) = {joint('the','cat','sat'):.4f}")
print(f"Tüm diziler üzerinden toplam = {total:.6f}  (1.0 olmalı)")

## 3. Bilgi kuramı: entropi, çapraz entropi, KL

**Shannon entropisi.**

$$H(p) = -\sum_x p(x) \log p(x).$$

Entropi, $p$'den alınan bir örneğin ortalama şaşırtıcılığıdır (surprisal) — onu en iyi kod altında kodlamak için gereken en küçük ortalama bit (log tabanı 2 ise) veya nat (doğal logaritma ise) sayısıdır.

Gerçek dağılım $p$ ile model $q$ arasındaki **çapraz entropi**:

$$H(p, q) = -\sum_x p(x) \log q(x).$$

**KL ıraksaması:**

$$D_{\mathrm{KL}}(p \,\|\, q) = H(p, q) - H(p) = \sum_x p(x) \log \frac{p(x)}{q(x)} \geq 0.$$

Bir dil modelini çapraz entropi kaybıyla eğitmek, $D_{\mathrm{KL}}(p_{\mathrm{veri}} \,\|\, p_\theta)$'yı en küçüklemekle tam olarak eşdeğerdir; çünkü $H(p_{\mathrm{veri}})$ $\theta$'ya bağlı değildir. Bu özdeşliğe sonraki her hafta geri döneceğiz.

In [ ]:
def entropy(p, base=2, eps=1e-12):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log(p + eps)) / np.log(base)

def cross_entropy(p, q, base=2, eps=1e-12):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    return -np.sum(p * np.log(q + eps)) / np.log(base)

def kl_divergence(p, q, base=2, eps=1e-12):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    return np.sum(p[mask] * np.log((p[mask] + eps) / (q[mask] + eps))) / np.log(base)

# Akıl sağlığı denetimleri
p = np.array([0.25, 0.25, 0.25, 0.25])      # 4 sembol üzerinde düzgün
q = np.array([0.5, 0.25, 0.125, 0.125])      # çarpık
print(f"H(p)        = {entropy(p):.4f} bit  (beklenen: log2(4) = 2.0)")
print(f"H(q)        = {entropy(q):.4f} bit  (beklenen: 1.75)")
print(f"H(p, q)     = {cross_entropy(p, q):.4f} bit")
print(f"KL(p || q)  = {kl_divergence(p, q):.4f} bit")
print(f"KL(p || p)  = {kl_divergence(p, p):.4f} bit  (0 olmalı)")

## 4. İngilizcenin deneysel entropisi

Shannon (1951), İngilizceyi karakter başına kabaca 1,0–1,5 bit olarak tahmin etmiştir — bu, 27 sembol üzerinde düzgün bir dağılımın gerektirdiği 4,76 bitin çok altındadır. Sıkıştırma algoritmalarının yararlandığı şey bu boşluktur. Burada tek-gram (unigram) bir tahmini yeniden üretiyoruz; daha zengin n-gram ve sinirsel tahminler 5–9. haftalarda gelecek.

In [ ]:
# Küçük bir İngilizce örnek. Gerçek bir derste WikiText-103 veya Brown derlemiyle değiştirin.
TEXT = (
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs. "
    "How vexingly quick daft zebras jump! "
    "Sphinx of black quartz, judge my vow. "
    "The five boxing wizards jump quickly. " * 50
).lower()

# Karakter düzeyinde unigram entropisi
char_counts = Counter(c for c in TEXT if c.isalpha() or c == ' ')
n_chars = sum(char_counts.values())
char_probs = np.array([c / n_chars for c in char_counts.values()])
H_char = entropy(char_probs)
print(f"Sözlük boyutu            : {len(char_counts)}")
print(f"Karakter başına unigram H: {H_char:.3f} bit")
print(f"Düzgün dağılım tabanı    : {np.log2(len(char_counts)):.3f} bit")
print(f"Shannon'ın tahmini       : karakter başına 1.0 – 1.5 bit (tam bağlamla)")
print()
print("En sık karakterler:")
for ch, c in char_counts.most_common(8):
    print(f"  {ch!r:5} {c:6d}  p={c/n_chars:.4f}")

In [ ]:
# Sözcük düzeyinde unigram entropisi
words = TEXT.split()
word_counts = Counter(words)
word_probs = np.array([c / len(words) for c in word_counts.values()])
H_word = entropy(word_probs)
print(f"Sözlük boyutu (sözcük)  : {len(word_counts)}")
print(f"Sözcük başına unigram H : {H_word:.3f} bit")
print(f"Düzgün dağılım tabanı   : {np.log2(len(word_counts)):.3f} bit")

# Zipf eğrisini görselleştir — NLP'de tekrar tekrar karşılaşılan bir motif.
freqs = sorted(word_counts.values(), reverse=True)
ranks = np.arange(1, len(freqs) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].loglog(ranks, freqs, marker='o', linestyle='-')
ax[0].set(xlabel='sıra (log)', ylabel='sıklık (log)', title='Zipf grafiği — sözcük sıklığı ve sıra')
ax[1].bar(range(min(10, len(char_counts))), [c for _, c in char_counts.most_common(10)])
ax[1].set_xticks(range(min(10, len(char_counts))))
ax[1].set_xticklabels([ch if ch != ' ' else '␣' for ch, _ in char_counts.most_common(10)])
ax[1].set(title='En sık 10 karakter', ylabel='sayı')
plt.tight_layout(); plt.show()

## 5. Karşılıklı bilgi

İki rastgele değişken $X, Y$ için:

$$I(X; Y) = \sum_{x, y} p(x, y) \log \frac{p(x, y)}{p(x) p(y)} = D_{\mathrm{KL}}(p(x, y) \,\|\, p(x) p(y)).$$

$I(X; Y) = 0$ ancak ve ancak $X \perp Y$ ise. NLP'de bir (sözcük, bağlam) çiftinin **noktasal karşılıklı bilgisi** (PMI), dağılımsal anlambilimin temelidir — 4. haftada word2vec'in örtük olarak kaydırılmış bir PMI matrisini çarpanlarına ayırdığını göreceğiz (Levy ve Goldberg, 2014).

In [ ]:
def pmi(p_xy, p_x, p_y, eps=1e-12):
    return np.log2((p_xy + eps) / (p_x * p_y + eps))

# Oyuncak örnek: r korelasyonlu iki madeni para
def joint_correlated(r):
    # P(00) = P(11) = (1+r)/4, P(01) = P(10) = (1-r)/4
    p = np.array([[(1+r)/4, (1-r)/4],
                  [(1-r)/4, (1+r)/4]])
    return p

for r in [0.0, 0.5, 0.9]:
    p_xy = joint_correlated(r)
    p_x = p_xy.sum(axis=1)
    p_y = p_xy.sum(axis=0)
    I = sum(p_xy[i,j] * pmi(p_xy[i,j], p_x[i], p_y[j])
            for i in range(2) for j in range(2))
    print(f"korelasyon r = {r:.1f}  ->  I(X;Y) = {I:.4f} bit")

## 6. Alıştırmalar

1. **KL'yi entropi ve çapraz entropiden türetin.** $D_{\mathrm{KL}}(p \| q) = H(p, q) - H(p)$ eşitliğini doğrudan tanımlardan gösterin.
2. **En çok olabilirlik ⇔ çapraz entropiyi en küçükleme.** Deneysel dağılım $\hat p$ için $H(\hat p, q_\theta)$'yı $\theta$ üzerinden en küçüklemenin, veri log-olabilirliği $\sum_i \log q_\theta(x_i)$'yi en büyüklemeye eşdeğer olduğunu gösterin.
3. **Türkçe ve İngilizce entropisi.** Türkçe sondan eklemeli bir dildir — tek bir sözcük, İngilizcenin bir öbekle anlattığını kodlayabilir. Karakter düzeyindeki entropinin Türkçe için daha yüksek mi yoksa daha düşük mü olduğunu tahmin edin, ardından küçük bir derlem üzerinde doğrulayın.
4. **KL'nin asimetrisi.** $D_{\mathrm{KL}}(p \| q) \neq D_{\mathrm{KL}}(q \| p)$ olacak şekilde $p, q$ oluşturun. Bu asimetrinin varyasyonel çıkarım için neden önemli olduğunu açıklayın (Hafta 9–10).

In [ ]:
# 3. alıştırma için başlangıç: karakter düzeyinde H(İngilizce) - H(Türkçe) tahmini.
ENGLISH = TEXT
TURKISH = ("İnsan hakları evrenseldir. " * 50 +
           "Türkçe sondan eklemeli bir dildir; "
           "tek bir kelime tüm bir cümleye karşılık gelebilir. " * 50).lower()

def char_entropy(s):
    counts = Counter(c for c in s if c.isalpha() or c == ' ')
    n = sum(counts.values())
    probs = np.array([c / n for c in counts.values()])
    return entropy(probs), len(counts)

H_en, V_en = char_entropy(ENGLISH)
H_tr, V_tr = char_entropy(TURKISH)
print(f"İngilizce: H = {H_en:.3f} bit, |V| = {V_en}")
print(f"Türkçe   : H = {H_tr:.3f} bit, |V| = {V_tr}")
print("\nNot: Türkçe ek harfler (ç, ğ, ı, ö, ş, ü) kullanır; bu da |V|'yi büyütür.")
print("H'nin artıp artmayacağı, bu harflerin ne kadar düzgün dağılımla kullanıldığına bağlıdır.")

---

## Gelecek Hafta

Hafta 2 — Metin İşleme ve Tokenizasyon. Karakterlerden tokenlara geçiyor; BPE, WordPiece ve Unigram dil modelini sıfırdan kuruyoruz.